# GS25 AI Assistant — Train tren Kaggle

**Checklist truoc khi chay:**
- Settings -> Accelerator -> **GPU P100** 
- Settings -> Internet -> **ON**
- Add-ons -> Secrets -> them `HF_TOKEN`
- Input -> them dataset `gs25_qa.jsonl`

Sau do bam **Run All** va cho doi.

In [ ]:
# =============================================================
# CELL 1: Kiem tra GPU va moi truong
# =============================================================
import subprocess, sys, os

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())

r2 = subprocess.run(['python', '--version'], capture_output=True, text=True)
print('Python:', r2.stdout.strip())

# Kiem tra internet
try:
    import urllib.request
    urllib.request.urlopen('https://github.com', timeout=5)
    print('Internet: OK')
except:
    print('Internet: FAILED -- Bat Internet trong Settings truoc!')

print('Input files:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else '(trong)')

In [ ]:
# =============================================================
# CELL 2: Cai PyTorch dung version cho P100 (sm_60)
# P100 can CUDA 11.x -- PyTorch >= 2.3 khong ho tro sm_60 nua
# =============================================================
import subprocess, sys

# Kiem tra torch hien tai
try:
    import torch
    if torch.cuda.is_available():
        cap = torch.cuda.get_device_capability(0)
        print(f'Current torch: {torch.__version__}, CUDA cap: {cap}')
        if cap[0] < 7:  # P100 = 6.0, can cai lai
            need_reinstall = True
        else:
            need_reinstall = False
    else:
        need_reinstall = True
except:
    need_reinstall = True

if need_reinstall:
    print('Cai lai PyTorch 2.2.2 + CUDA 11.8 cho P100...')
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], capture_output=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.2.2', 'torchvision==0.17.2',
        '--index-url', 'https://download.pytorch.org/whl/cu118'
    ])
    print('Xong! Can RESTART KERNEL --> Run --> Restart & Clear Output --> Run All')
else:
    print('PyTorch OK, khong can cai lai')

In [ ]:
# =============================================================
# CELL 3: Cai cac thu vien can thiet (numpy compat voi torch 2.2)
# =============================================================
import subprocess, sys

pkgs = [
    'numpy==1.26.4',          # torch 2.2 + cu118 can numpy 1.x
    'transformers==4.44.2',   # phien ban on dinh
    'tokenizers==0.19.1',
    'huggingface_hub==0.24.6',
    'datasets==2.21.0',
    'tiktoken',
    'sentencepiece',
    'accelerate==0.33.0',
]

for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], capture_output=True)
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')

# Verify
import importlib
print()
for mod in ['torch', 'transformers', 'numpy', 'datasets']:
    try:
        m = importlib.import_module(mod)
        print(f'  {mod}: {getattr(m, "__version__", "ok")}')
    except Exception as e:
        print(f'  {mod}: ERROR - {e}')

In [ ]:
# =============================================================
# CELL 4: Doc HF Token tu Kaggle Secrets
# =============================================================
HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Token tu Kaggle Secrets: OK')
except Exception as e:
    print(f'Secrets error: {e}')
    HF_TOKEN = input('Nhap HF_TOKEN (hf_...): ').strip()

if HF_TOKEN and HF_TOKEN.startswith('hf_'):
    print(f'Token: {HF_TOKEN[:12]}...{HF_TOKEN[-4:]}')
else:
    print('WARNING: Token co the sai dinh dang')

In [ ]:
# =============================================================
# CELL 5: Clone MiniMind
# =============================================================
import os

WORK = '/kaggle/working'
MM_DIR = f'{WORK}/minimind'

if not os.path.exists(MM_DIR):
    ret = os.system(f'git clone --depth 1 https://github.com/jingyaogong/minimind.git {MM_DIR}')
    print('Clone:', 'OK' if ret == 0 else 'FAILED')
else:
    print('MiniMind da co san')

# Liet ke cac script train
train_scripts = []
for root, _, files in os.walk(MM_DIR):
    for f in files:
        if 'train' in f.lower() and f.endswith('.py'):
            train_scripts.append(os.path.join(root, f).replace(MM_DIR+'/', ''))
print('Train scripts:', train_scripts)

In [ ]:
# =============================================================
# CELL 6: Tai tokenizer tu HuggingFace
# =============================================================
import os
from huggingface_hub import snapshot_download, login

MM_DIR = '/kaggle/working/minimind'
TOKENIZER_DIR = f'{MM_DIR}/model'
os.makedirs(TOKENIZER_DIR, exist_ok=True)

# Dang nhap HF
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF Login: OK')

# Tai tokenizer (chi tai file nho, khong tai weights)
TOKENIZER_FILES = [
    'tokenizer.json',
    'tokenizer_config.json',
    'special_tokens_map.json',
]

already = [f for f in TOKENIZER_FILES if os.path.exists(f'{TOKENIZER_DIR}/{f}')]
if len(already) == len(TOKENIZER_FILES):
    print('Tokenizer da co san:', already)
else:
    print('Tai tokenizer tu jingyaogong/minimind-3o...')
    try:
        from huggingface_hub import hf_hub_download
        for fname in TOKENIZER_FILES:
            try:
                path = hf_hub_download(
                    repo_id='jingyaogong/minimind-3o',
                    filename=fname,
                    token=HF_TOKEN,
                    local_dir=TOKENIZER_DIR
                )
                print(f'  Downloaded: {fname}')
            except Exception as e:
                print(f'  Skip {fname}: {e}')
    except Exception as e:
        print(f'HF download error: {e}')

print('Tokenizer files:', os.listdir(TOKENIZER_DIR))

In [ ]:
# =============================================================
# CELL 7: Patch trainer_utils.py -- fix tokenizer path
# Van de: '../model' la relative path, transformers moi reject
# Fix: doi thanh absolute path
# =============================================================
import os

MM_DIR = '/kaggle/working/minimind'
utils_path = f'{MM_DIR}/trainer/trainer_utils.py'

with open(utils_path, 'r', encoding='utf-8') as f:
    content = f.read()

original = content

# Fix 1: Doi default tokenizer_path thanh absolute path
content = content.replace(
    "tokenizer_path='../model'",
    f"tokenizer_path='{MM_DIR}/model'"
)

# Fix 2: Them local_files_only vao AutoTokenizer.from_pretrained neu can
# de tranh goi HuggingFace API voi path chua dau '/'
content = content.replace(
    'tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)',
    'tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=os.path.isabs(tokenizer_path) and os.path.exists(tokenizer_path))'
)

# Dam bao import os co san trong file
if 'import os' not in content[:500]:
    content = 'import os\n' + content

if content != original:
    with open(utils_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print('Patched trainer_utils.py OK')
else:
    print('Khong tim thay doan can patch -- xem noi dung:')
    for i, l in enumerate(content.split('\n')[115:130], 116):
        print(f'{i}: {l}')

In [ ]:
# =============================================================
# CELL 8: Chuan bi dataset GS25
# =============================================================
import json, os

MM_DIR = '/kaggle/working/minimind'
os.makedirs(f'{MM_DIR}/dataset', exist_ok=True)
SFT_PATH = f'{MM_DIR}/dataset/sft_gs25.jsonl'

# Tim file .jsonl trong /kaggle/input
src_file = None
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.jsonl'):
            src_file = os.path.join(root, f)
            break

if src_file:
    print(f'Tim thay: {src_file}')
    raw = open(src_file, encoding='utf-8').readlines()
else:
    print('Khong tim thay file .jsonl trong Input -- dung dataset mau')
    # Dataset mau nho de test pipeline
    raw = [
        json.dumps({'instruction': 'Part-time toi da lam bao nhieu gio moi tuan?', 'input': '', 'output': 'Part-time (STPT) tai GS25 bi gioi han toi da 23 gio/tuan va 91 gio/thang.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Cau chao khach chuan cua GS25?', 'input': '', 'output': 'Khi khach vao: GS25 xin chao! Khi khach ra: GS25 cam on va hen gap lai!'}, ensure_ascii=False),
        json.dumps({'instruction': 'Ca dem GS25 may gio?', 'input': '', 'output': 'Ca dem tai GS25 tu 22:00 den 06:00 sang hom sau (ca 22-6).'}, ensure_ascii=False),
        json.dumps({'instruction': 'Chu ky luong GS25 tu ngay may?', 'input': '', 'output': 'Chu ky luong GS25 tu ngay 26 thang truoc den ngay 25 thang hien tai.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Full-time GS25 lam bao nhieu gio/tuan?', 'input': '', 'output': 'Full-time (STFT) chuan 48 gio/tuan, tuong duong 6 ca 8 tieng va 1 ngay nghi OFF.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Ca dem co phu cap khong?', 'input': '', 'output': 'Co. Ca dem 22-6 duoc tinh phu cap x1.3 so voi ca ngay theo Dieu 98 Bo luat Lao dong 2019.'}, ensure_ascii=False),
        json.dumps({'instruction': 'Lam the nao de doi ca?', 'input': '', 'output': 'NV vao Lich ca -> bam Doi ca -> chon nguoi doi -> doi phuong xac nhan -> SM duyet.'}, ensure_ascii=False),
        json.dumps({'instruction': 'SM co quyen gi?', 'input': '', 'output': 'SM quan ly lich ca, cham cong, duyet C&B va doi ca trong pham vi cua hang duoc gan sm_id. Khong co quyen trang Nhan vien va Nhat ky.'}, ensure_ascii=False),
    ]

# Convert sang format conversations cua MiniMind
converted = []
for line in raw:
    item = json.loads(line.strip())
    if 'conversations' in item:
        converted.append(item)
    else:
        converted.append({'conversations': [
            {'role': 'system', 'content': 'Ban la AI tro ly nghiep vu GS25 Viet Nam.', 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
            {'role': 'user', 'content': item.get('instruction',''), 'reasoning_content': '', 'tools': '', 'tool_calls': ''},
            {'role': 'assistant', 'content': item.get('output',''), 'reasoning_content': '', 'tools': '', 'tool_calls': ''}
        ]})

with open(SFT_PATH, 'w', encoding='utf-8') as f:
    for d in converted:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print(f'Dataset: {len(converted)} samples -> {SFT_PATH}')
s = converted[0]['conversations']
print(f'Sample: Q={s[1]["content"][:50]} | A={s[2]["content"][:50]}')

In [ ]:
# =============================================================
# CELL 9: Kiem tra tokenizer load duoc khong
# =============================================================
from transformers import AutoTokenizer
import os

MM_DIR = '/kaggle/working/minimind'
TOKENIZER_DIR = f'{MM_DIR}/model'

print('Files trong model/:', os.listdir(TOKENIZER_DIR))

try:
    tok = AutoTokenizer.from_pretrained(TOKENIZER_DIR, local_files_only=True)
    print('Tokenizer load OK')
    print('Vocab size:', tok.vocab_size)
    test = tok.encode('GS25 xin chao!')
    print('Test encode:', test[:10], '...')
except Exception as e:
    print(f'Tokenizer ERROR: {e}')
    print('Can download tokenizer truoc (chay lai Cell 6)')

In [ ]:
# =============================================================
# CELL 10: CHAY TRAIN
# =============================================================
import subprocess, os, time

MM_DIR = '/kaggle/working/minimind'
os.chdir(MM_DIR)

SAVE_DIR = 'out/gs25_sft'
os.makedirs(SAVE_DIR, exist_ok=True)

cmd = [
    'python', 'trainer/train_full_sft.py',
    '--data_path',          'dataset/sft_gs25.jsonl',
    '--save_dir',           SAVE_DIR,
    '--epochs',             '15',
    '--batch_size',         '4',
    '--learning_rate',      '3e-5',
    '--device',             'cuda',
    '--dtype',              'float16',
    '--max_seq_len',        '512',
    '--from_weight',        'none',
    '--log_interval',       '5',
]

print('Command:', ' '.join(cmd))
print('='*60)
start = time.time()
result = subprocess.run(cmd, text=True)
elapsed = (time.time() - start) / 60
print(f'Time: {elapsed:.1f} min')
if result.returncode == 0:
    print('DONE! Files:', os.listdir(f'{MM_DIR}/{SAVE_DIR}'))
else:
    print('ERROR -- returncode:', result.returncode)

In [ ]:
# =============================================================
# CELL 11: Upload model len HuggingFace
# =============================================================
from huggingface_hub import HfApi
import os

MM_DIR = '/kaggle/working/minimind'
MODEL_DIR = f'{MM_DIR}/out/gs25_sft'

api = HfApi(token=HF_TOKEN)

try:
    info = api.whoami()
    username = info['name']
    print(f'HF user: {username}')
except Exception as e:
    print(f'HF auth error: {e}')
    username = input('Nhap HF username: ').strip()

REPO_ID = f'{username}/gs25-assistant'

files = os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else []
if not files:
    print('Model chua co -- chay Cell 10 truoc')
else:
    print(f'Files: {files}')
    api.create_repo(repo_id=REPO_ID, exist_ok=True)
    api.upload_folder(folder_path=MODEL_DIR, repo_id=REPO_ID, repo_type='model')
    print(f'Upload OK: https://huggingface.co/{REPO_ID}')
    API_URL = f'https://api-inference.huggingface.co/models/{REPO_ID}'
    print()
    print('=== THEM VAO .env CUA DU AN GS25 ===')
    print(f'VITE_GS25_AI_MODEL_URL={API_URL}')
    print(f'VITE_HF_TOKEN={HF_TOKEN}')
    print('===================================')

In [ ]:
# =============================================================
# CELL 12 (Tuy chon): Download model ve may
# =============================================================
import shutil, os
from IPython.display import FileLink, display

MM_DIR = '/kaggle/working/minimind'
MODEL_DIR = f'{MM_DIR}/out/gs25_sft'
ARCHIVE = '/kaggle/working/gs25_model.tar.gz'

if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    shutil.make_archive('/kaggle/working/gs25_model', 'gztar', MODEL_DIR)
    size = os.path.getsize(ARCHIVE) / 1e6
    print(f'Archive: {size:.1f} MB')
    display(FileLink('/kaggle/working/gs25_model.tar.gz'))
else:
    print('Model chua co -- chay Cell 10 truoc')